# Final Method Evaluation: Two-Stage Gaussian Filtering with PCA_10D

This notebook evaluates the final chosen method from `gaussian_filtering_advanced` on the test set.

**Method Configuration:**
- Stage 1: 3D Gaussian on band-wise reconstruction losses (LF, BP, HF)
- Stage 2: PCA-reduced 10D latent space
- Confidence Stage 1: 0.95
- Confidence Stage 2: 0.70

**Requirements:**
- Google Colab A100 GPU
- Google Drive with:
  - `/content/drive/MyDrive/LRdataset/test_balanced_npz.zip`
  - `/content/drive/MyDrive/LRdataset/data_split (1).json`
  - `/content/drive/MyDrive/liveness_checkpoints/stage2/` (saved model weights)

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q numpy scipy
!pip install -q matplotlib seaborn
!pip install -q scikit-learn
!pip install -q tqdm

In [ ]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import signal, stats
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, precision_recall_fscore_support
)
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Extract Test Data

In [ ]:
# Extract test data
zip_path = '/content/drive/MyDrive/LRdataset/test_balanced_npz.zip'
extract_path = '/content/test_balanced_npz/'

if not os.path.exists(extract_path):
    print("Extracting test data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f"Extracted to {extract_path}")
else:
    print(f"Test data already extracted at {extract_path}")

## 3. Model Architecture Definitions

In [ ]:
# Butterworth Filter Bank
class ButterworthFilterBank:
    def __init__(self, fps=30, fc_low=2.0, fc_high=8.0, order=4):
        self.fps = fps
        nyquist = fps / 2.0
        wn_low = fc_low / nyquist
        wn_high = fc_high / nyquist
        self.sos_lf = signal.butter(order, wn_low, btype='lowpass', output='sos')
        self.sos_bp = signal.butter(order, [wn_low, wn_high], btype='bandpass', output='sos')
        self.sos_hf = signal.butter(order, wn_high, btype='highpass', output='sos')

    def apply(self, x):
        T, K, D = x.shape
        x_flat = x.transpose(1, 2, 0).reshape(K * D, T)
        x_lf_flat = signal.sosfiltfilt(self.sos_lf, x_flat, axis=1)
        x_bp_flat = signal.sosfiltfilt(self.sos_bp, x_flat, axis=1)
        x_hf_flat = signal.sosfiltfilt(self.sos_hf, x_flat, axis=1)
        x_lf = x_lf_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_bp = x_bp_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_hf = x_hf_flat.reshape(K, D, T).transpose(2, 0, 1)
        return x_lf, x_bp, x_hf


# TCN Block
class TCNBlock(nn.Module):
    def __init__(self, C_in, C_out, dilation=1, kernel_size=3):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(C_in, C_out, kernel_size=kernel_size, padding=padding, dilation=dilation)
        self.gn = nn.GroupNorm(1, C_out)
        self.act = nn.SiLU()
        self.res = nn.Conv1d(C_in, C_out, 1) if C_in != C_out else nn.Identity()

    def forward(self, x):
        y = self.act(self.gn(self.conv(x)))
        return y + self.res(x)


# Single-Band VAE
class SingleBandVAE(nn.Module):
    def __init__(self, C_in, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        self.C_in = C_in
        self.C_h = C_h
        self.C_z = C_z

        self.enc_inp = nn.Conv1d(C_in, C_h, 1)
        enc_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in dilations]
        self.encoder = nn.Sequential(*enc_blocks)
        self.enc_out = nn.Conv1d(C_h, C_z * 2, 1)

        self.dec_inp = nn.Conv1d(C_z, C_h, 1)
        dec_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in reversed(dilations)]
        self.decoder = nn.Sequential(*dec_blocks)
        self.dec_out = nn.Conv1d(C_h, C_in, 1)

    def encode(self, x):
        h = self.encoder(self.enc_inp(x))
        mu, logvar = torch.chunk(self.enc_out(h), 2, dim=1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder(self.dec_inp(z))
        return self.dec_out(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


# Band-Split VAE
class BandSplitVAE(nn.Module):
    def __init__(self, C_in_per_band, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        self.C_in_per_band = C_in_per_band
        self.vae_lf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_bp = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_hf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.register_parameter('fusion_weights', nn.Parameter(torch.ones(3) / 3.0))

    def forward(self, x_lf, x_bp, x_hf):
        x_hat_lf, mu_lf, logvar_lf = self.vae_lf(x_lf)
        x_hat_bp, mu_bp, logvar_bp = self.vae_bp(x_bp)
        x_hat_hf, mu_hf, logvar_hf = self.vae_hf(x_hf)

        weights = F.softmax(self.fusion_weights, dim=0)
        x_hat_fused = weights[0] * x_hat_lf + weights[1] * x_hat_bp + weights[2] * x_hat_hf

        recons = {'lf': x_hat_lf, 'bp': x_hat_bp, 'hf': x_hat_hf}
        mus = {'lf': mu_lf, 'bp': mu_bp, 'hf': mu_hf}
        logvars = {'lf': logvar_lf, 'bp': logvar_bp, 'hf': logvar_hf}

        return recons, mus, logvars, x_hat_fused


print("✓ Model architectures defined")

## 4. Feature Engineering Functions

In [ ]:
def compute_velocity(x):
    """Compute velocity using central differences"""
    v = np.zeros_like(x)
    v[1:-1] = (x[2:] - x[:-2]) / 2.0
    v[0] = x[1] - x[0]
    v[-1] = x[-1] - x[-2]
    return v


def compute_acceleration(v):
    """Compute acceleration from velocity"""
    a = np.zeros_like(v)
    a[1:-1] = (v[2:] - v[:-2]) / 2.0
    a[0] = v[1] - v[0]
    a[-1] = v[-1] - v[-2]
    return a


def compute_angle(x):
    """Compute angle from (x, y) coordinates"""
    theta = np.arctan2(x[:, :, 1], x[:, :, 0])
    return theta[..., np.newaxis]


def compute_angle_rate(theta):
    """Compute angular velocity"""
    dtheta = np.zeros_like(theta)
    dtheta[1:-1] = (theta[2:] - theta[:-2]) / 2.0
    dtheta[0] = theta[1] - theta[0]
    dtheta[-1] = theta[-1] - theta[-2]
    return dtheta


def extract_features_full(x):
    """Extract full features: position + velocity + acceleration + angle + angle_rate"""
    v = compute_velocity(x)
    a = compute_acceleration(v)
    theta = compute_angle(x)
    dtheta = compute_angle_rate(theta)
    features = np.concatenate([x, v, a, theta, dtheta], axis=-1)
    T, K, F = features.shape
    return features.transpose(1, 2, 0).reshape(K * F, T)


print("✓ Feature engineering functions defined")

## 5. Dataset Classes

In [ ]:
class Stage2Dataset(Dataset):
    """Dataset for Stage 2 with filename-based matching"""
    
    def __init__(self, split_json, split_name, data_root, T_fixed=300):
        self.data_root = data_root
        self.T_fixed = T_fixed
        self.filter_bank = ButterworthFilterBank(fps=30, fc_low=2.0, fc_high=8.0, order=4)

        # Build file index (filename -> full_path)
        print(f"Building file index from {data_root}...")
        self.file_index = {}
        for root, dirs, files in os.walk(data_root):
            for file in files:
                if file.endswith('.npz'):
                    full_path = os.path.join(root, file)
                    self.file_index[file] = full_path
        print(f"Found {len(self.file_index)} files in directory")

        # Load split
        with open(split_json, 'r') as f:
            data = json.load(f)
        
        if split_name not in data:
            raise KeyError(f"Split '{split_name}' not found in JSON. Available: {list(data.keys())}")
        
        split = data[split_name]

        # Match files using filename only
        self.samples = []
        missing_files = []

        for file_path in split['real']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.samples.append((self.file_index[filename], 0))
            else:
                missing_files.append(filename)

        for file_path in split['fake']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.samples.append((self.file_index[filename], 1))
            else:
                missing_files.append(filename)

        if missing_files:
            print(f"⚠ Warning: {len(missing_files)} files not found in directory")
            print(f"  First few missing: {missing_files[:5]}")

        print(f"Loaded {len(self.samples)} samples for split '{split_name}'")
        n_real = sum(1 for _, label in self.samples if label == 0)
        n_fake = sum(1 for _, label in self.samples if label == 1)
        print(f"  Real: {n_real}, Fake: {n_fake}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        data = np.load(file_path)
        x = data['lips']

        # Pad or crop to T_fixed
        T = x.shape[0]
        if T < self.T_fixed:
            pad = self.T_fixed - T
            x = np.pad(x, ((0, pad), (0, 0), (0, 0)), mode='edge')
        else:
            x = x[:self.T_fixed]

        # Apply frequency decomposition
        x_lf, x_bp, x_hf = self.filter_bank.apply(x)

        # Extract full features
        f_lf = extract_features_full(x_lf)
        f_bp = extract_features_full(x_bp)
        f_hf = extract_features_full(x_hf)

        return (
            torch.from_numpy(f_lf).float(),
            torch.from_numpy(f_bp).float(),
            torch.from_numpy(f_hf).float(),
            torch.tensor(label, dtype=torch.long)
        )


print("✓ Dataset class defined")

## 6. Gaussian Filtering Functions

In [ ]:
def fit_gaussian_multivariate(data):
    """Fit multivariate Gaussian and return mean, covariance"""
    mu = np.mean(data, axis=0)
    cov = np.cov(data, rowvar=False)
    return mu, cov


def is_within_confidence_multivariate(x, mu, cov, confidence):
    """Check if point is within confidence ellipsoid for multivariate Gaussian"""
    try:
        diff = x - mu
        cov_inv = np.linalg.inv(cov + 1e-6 * np.eye(len(mu)))
        mahal_dist_sq = diff @ cov_inv @ diff

        dim = len(mu)
        threshold = stats.chi2.ppf(confidence, dim)

        return mahal_dist_sq <= threshold
    except np.linalg.LinAlgError:
        sigma = np.sqrt(np.diag(cov))
        z_score = stats.norm.ppf((1 + confidence) / 2)
        within = np.all(np.abs(x - mu) <= z_score * sigma)
        return within


print("✓ Gaussian filtering functions defined")

## 7. Feature Extraction Functions

In [ ]:
@torch.no_grad()
def compute_bandwise_reconstruction_losses(model, loader, device):
    """Compute reconstruction loss for each band separately"""
    model.eval()

    all_losses_lf = []
    all_losses_bp = []
    all_losses_hf = []
    all_labels = []

    for x_lf, x_bp, x_hf, labels in tqdm(loader, desc="Extracting band losses"):
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)

        # Forward pass
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)

        # Compute loss for each band separately
        batch_size = x_lf.size(0)

        for i in range(batch_size):
            # LF band reconstruction loss
            loss_lf = F.mse_loss(recons['lf'][i], x_lf[i])
            all_losses_lf.append(loss_lf.item())

            # BP band reconstruction loss
            loss_bp = F.mse_loss(recons['bp'][i], x_bp[i])
            all_losses_bp.append(loss_bp.item())

            # HF band reconstruction loss
            loss_hf = F.mse_loss(recons['hf'][i], x_hf[i])
            all_losses_hf.append(loss_hf.item())

            all_labels.append(labels[i].item())

    # Stack into (N, 3) array
    losses_3d = np.stack([all_losses_lf, all_losses_bp, all_losses_hf], axis=1)
    labels = np.array(all_labels)

    return losses_3d, labels


@torch.no_grad()
def extract_latent_vectors(model, loader, device):
    """Extract latent vectors (concatenated mu from all bands)"""
    model.eval()

    all_latents = []
    all_labels = []

    for x_lf, x_bp, x_hf, labels in tqdm(loader, desc="Extracting latents"):
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)

        # Forward pass
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)

        # Get mean of latent vectors across time dimension
        z_lf = mus['lf'].mean(dim=2)  # (B, C_z)
        z_bp = mus['bp'].mean(dim=2)  # (B, C_z)
        z_hf = mus['hf'].mean(dim=2)  # (B, C_z)

        z_concat = torch.cat([z_lf, z_bp, z_hf], dim=1)  # (B, C_z * 3)

        all_latents.append(z_concat.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_latents = np.vstack(all_latents)
    all_labels = np.array(all_labels)

    return all_latents, all_labels


print("✓ Feature extraction functions defined")

## 8. Configuration

In [ ]:
# Configuration (from gaussian_filtering_advanced best result)
PCA_DIM = 10
CONFIDENCE_STAGE1 = 0.95
CONFIDENCE_STAGE2 = 0.70
T_FIXED = 300

# Paths
SPLIT_JSON = '/content/drive/MyDrive/LRdataset/data_split (1).json'
DATA_ROOT = '/content/test_balanced_npz/'
CHECKPOINT_DIR = '/content/drive/MyDrive/liveness_checkpoints/stage2/'
OUTPUT_DIR = '/content/drive/MyDrive/liveness_checkpoints/final_evaluation/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model config (Full features)
K = 40
F_dim = 8  # Full features
C_in_per_band = K * F_dim  # 320
C_h = 48
C_z = 12

print(f"Configuration:")
print(f"  PCA dimensions: {PCA_DIM}")
print(f"  Stage 1 confidence: {CONFIDENCE_STAGE1}")
print(f"  Stage 2 confidence: {CONFIDENCE_STAGE2}")
print(f"  T_fixed: {T_FIXED}")
print(f"  C_in_per_band: {C_in_per_band}")
print(f"  Device: {device}")

## 9. Load Datasets

In [ ]:
# Load training data (for fitting Gaussians)
print("\nLoading training dataset...")
train_dataset = Stage2Dataset(
    split_json=SPLIT_JSON,
    split_name='stage2_train',
    data_root=DATA_ROOT,
    T_fixed=T_FIXED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Load test data
print("\nLoading test dataset...")
test_dataset = Stage2Dataset(
    split_json=SPLIT_JSON,
    split_name='final_test',
    data_root=DATA_ROOT,
    T_fixed=T_FIXED
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\nDataset loaded:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

## 10. Load Trained Model

In [ ]:
# Find the best checkpoint
print("\nLooking for model checkpoint...")
checkpoint_path = None

# Try different naming patterns
possible_names = [
    'method1_margin_full_best.pt',
    'stage2_method1_best.pt',
    'method1_margin_simple_best.pt'
]

for name in possible_names:
    path = os.path.join(CHECKPOINT_DIR, name)
    if os.path.exists(path):
        checkpoint_path = path
        break

if checkpoint_path is None:
    print("Available checkpoints:")
    if os.path.exists(CHECKPOINT_DIR):
        for f in os.listdir(CHECKPOINT_DIR):
            if f.endswith('.pt'):
                print(f"  - {f}")
        # Use the first available .pt file
        pt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]
        if pt_files:
            checkpoint_path = os.path.join(CHECKPOINT_DIR, pt_files[0])
            print(f"\nUsing: {pt_files[0]}")
    else:
        raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

if checkpoint_path is None:
    raise FileNotFoundError("No checkpoint found!")

print(f"Loading checkpoint: {checkpoint_path}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Initialize model
model = BandSplitVAE(
    C_in_per_band=C_in_per_band,
    C_h=C_h,
    C_z=C_z,
    dilations=[1, 2, 4]
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✓ Model loaded from epoch {checkpoint.get('epoch', 'N/A')}")
print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 11. Extract Features from Training Data

In [ ]:
print("\nExtracting features from training data...")

# Extract band-wise reconstruction losses
train_losses_3d, train_labels = compute_bandwise_reconstruction_losses(
    model, train_loader, device
)

# Extract latent vectors
train_latents_36d, _ = extract_latent_vectors(model, train_loader, device)

print(f"\nTraining features extracted:")
print(f"  3D losses shape: {train_losses_3d.shape}")
print(f"  36D latents shape: {train_latents_36d.shape}")

# Filter for real samples only
train_real_mask = (train_labels == 0)
train_real_losses_3d = train_losses_3d[train_real_mask]
train_real_latents_36d = train_latents_36d[train_real_mask]

print(f"  Real training samples: {len(train_real_losses_3d)}")

## 12. Fit Stage 1 Gaussian (3D on losses)

In [ ]:
print("\nFitting Stage 1 Gaussian (3D band-wise losses)...")

loss_mu_3d, loss_cov_3d = fit_gaussian_multivariate(train_real_losses_3d)

print(f"  3D Loss mean: {loss_mu_3d}")
print(f"  3D Loss covariance shape: {loss_cov_3d.shape}")
print(f"  Covariance diagonal: {np.diag(loss_cov_3d)}")

## 13. Apply PCA and Fit Stage 2 Gaussian

In [ ]:
print(f"\nApplying PCA to reduce latent space to {PCA_DIM}D...")

# Extract test features
test_losses_3d, test_labels = compute_bandwise_reconstruction_losses(
    model, test_loader, device
)
test_latents_36d, _ = extract_latent_vectors(model, test_loader, device)

print(f"\nTest features extracted:")
print(f"  3D losses shape: {test_losses_3d.shape}")
print(f"  36D latents shape: {test_latents_36d.shape}")

# Apply PCA
pca = PCA(n_components=PCA_DIM)
train_latents_pca = pca.fit_transform(train_real_latents_36d)
test_latents_pca = pca.transform(test_latents_36d)

explained_var = np.sum(pca.explained_variance_ratio_)

print(f"\n  PCA explained variance: {explained_var:.4f}")
print(f"  PCA latent shapes: train={train_latents_pca.shape}, test={test_latents_pca.shape}")

# Fit Stage 2 Gaussian
print("\nFitting Stage 2 Gaussian (PCA-reduced latent space)...")
latent_mu_pca, latent_cov_pca = fit_gaussian_multivariate(train_latents_pca)
print(f"  PCA latent mean (first 5): {latent_mu_pca[:5]}")
print(f"  PCA latent cov shape: {latent_cov_pca.shape}")

## 14. Apply Two-Stage Filtering

In [ ]:
print("\nApplying two-stage filtering on test data...")

num_test = len(test_losses_3d)
predictions = []
stage1_pass_count = 0
stage2_pass_count = 0

for i in tqdm(range(num_test), desc="Filtering"):
    loss_vec = test_losses_3d[i]  # (3,)
    latent = test_latents_pca[i]  # (10,)

    # Stage 1: 3D Gaussian on band-wise reconstruction losses
    stage1_pass = is_within_confidence_multivariate(
        loss_vec, loss_mu_3d, loss_cov_3d, CONFIDENCE_STAGE1
    )

    if stage1_pass:
        stage1_pass_count += 1
        # Stage 2: Latent distribution filter
        stage2_pass = is_within_confidence_multivariate(
            latent, latent_mu_pca, latent_cov_pca, CONFIDENCE_STAGE2
        )

        if stage2_pass:
            stage2_pass_count += 1
            predictions.append(0)  # Real
        else:
            predictions.append(1)  # Fake
    else:
        predictions.append(1)  # Fake

predictions = np.array(predictions)

print(f"\nFiltering complete:")
print(f"  Stage 1 pass rate: {stage1_pass_count}/{num_test} ({100*stage1_pass_count/num_test:.2f}%)")
print(f"  Stage 2 pass rate: {stage2_pass_count}/{num_test} ({100*stage2_pass_count/num_test:.2f}%)")
print(f"  Classified as Real: {np.sum(predictions == 0)}/{num_test}")
print(f"  Classified as Fake: {np.sum(predictions == 1)}/{num_test}")

## 15. Compute Metrics

In [ ]:
# Compute metrics
cm = confusion_matrix(test_labels, predictions)
accuracy = accuracy_score(test_labels, predictions)
precision = precision_score(test_labels, predictions, zero_division=0)
recall = recall_score(test_labels, predictions, zero_division=0)
f1 = f1_score(test_labels, predictions, zero_division=0)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Print results
print(f"\n{'='*80}")
print(f"EVALUATION RESULTS")
print(f"{'='*80}")
print(f"Method: Two-Stage Gaussian Filtering (PCA_10D)")
print(f"Configuration: Stage1_Conf={CONFIDENCE_STAGE1}, Stage2_Conf={CONFIDENCE_STAGE2}")
print(f"{'='*80}")
print(f"")
print(f"Primary Metrics:")
print(f"  Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision:   {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall:      {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1-Score:    {f1:.4f}")
print(f"  Specificity: {specificity:.4f} ({specificity*100:.2f}%)")
print(f"")
print(f"Confusion Matrix:")
print(f"                Predicted")
print(f"              Real    Fake")
print(f"Actual Real   {tn:4d}    {fp:4d}")
print(f"       Fake   {fn:4d}    {tp:4d}")
print(f"")
print(f"{'='*80}")

## 16. Visualizations

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
ax1.set_title(f'Confusion Matrix\nAccuracy: {accuracy:.4f}, F1: {f1:.4f}',
             fontsize=14, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=12)
ax1.set_xlabel('Predicted Label', fontsize=12)

# Add percentages
for i in range(2):
    for j in range(2):
        percentage = cm_normalized[i, j] * 100
        ax1.text(j + 0.5, i + 0.75, f'({percentage:.1f}%)',
                ha='center', va='center', fontsize=11, color='gray')

# 2. Metrics Bar Chart
ax2 = fig.add_subplot(gs[0, 1])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
metrics_values = [accuracy, precision, recall, f1, specificity]
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
bars = ax2.bar(metrics_names, metrics_values, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_ylim([0, 1.0])
ax2.set_ylabel('Score', fontsize=12, fontweight='bold')
ax2.set_title('Performance Metrics', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, metrics_values):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Filtering Statistics
ax3 = fig.add_subplot(gs[0, 2])
stage1_pass_rate = stage1_pass_count / num_test
stage2_pass_rate = stage2_pass_count / num_test
classified_real = np.sum(predictions == 0) / len(predictions)
classified_fake = np.sum(predictions == 1) / len(predictions)

categories = ['Stage 1\nPass', 'Stage 2\nPass', 'Classified\nReal', 'Classified\nFake']
stats_values = [stage1_pass_rate, stage2_pass_rate, classified_real, classified_fake]
stats_colors = ['#2ca02c', '#ff7f0e', '#1f77b4', '#d62728']
bars2 = ax3.bar(categories, stats_values, color=stats_colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax3.set_ylim([0, 1.0])
ax3.set_ylabel('Rate', fontsize=12, fontweight='bold')
ax3.set_title('Filtering Statistics', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, stats_values):
    ax3.text(bar.get_x() + bar.get_width()/2., val + 0.02,
            f'{val:.2%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. Band-wise Loss Distribution (Real vs Fake)
ax4 = fig.add_subplot(gs[1, 0])
real_mask = test_labels == 0
fake_mask = test_labels == 1

band_names = ['LF', 'BP', 'HF']
x_pos = np.arange(len(band_names))
real_means = test_losses_3d[real_mask].mean(axis=0)
fake_means = test_losses_3d[fake_mask].mean(axis=0)
width = 0.35

ax4.bar(x_pos - width/2, real_means, width, label='Real', color='green', alpha=0.7)
ax4.bar(x_pos + width/2, fake_means, width, label='Fake', color='red', alpha=0.7)
ax4.set_xlabel('Frequency Band', fontsize=12)
ax4.set_ylabel('Mean Reconstruction Loss', fontsize=12)
ax4.set_title('Band-wise Loss Distribution', fontsize=14, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(band_names)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 5. PCA Latent Space (2D projection)
ax5 = fig.add_subplot(gs[1, 1])
real_latents = test_latents_pca[real_mask]
fake_latents = test_latents_pca[fake_mask]
ax5.scatter(real_latents[:, 0], real_latents[:, 1], c='green', alpha=0.5, label='Real', s=30)
ax5.scatter(fake_latents[:, 0], fake_latents[:, 1], c='red', alpha=0.5, label='Fake', s=30)
ax5.scatter(latent_mu_pca[0], latent_mu_pca[1], c='blue', marker='x', s=200, linewidths=3, label='Gaussian Center')
ax5.set_xlabel('PC1', fontsize=12)
ax5.set_ylabel('PC2', fontsize=12)
ax5.set_title('PCA Latent Space (First 2 PCs)', fontsize=14, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Summary Text
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_text = f"""
EVALUATION SUMMARY
{'='*45}

Method: Two-Stage Gaussian Filtering
        with PCA_10D

Configuration:
  Stage 1 Confidence: {CONFIDENCE_STAGE1}
  Stage 2 Confidence: {CONFIDENCE_STAGE2}
  PCA Dimensions:     {PCA_DIM}
  Explained Variance: {explained_var:.4f}

Dataset:
  Test samples:       {len(test_labels)}
  Real samples:       {(test_labels == 0).sum()}
  Fake samples:       {(test_labels == 1).sum()}

Performance:
  Accuracy:           {accuracy:.4f}
  Precision:          {precision:.4f}
  Recall:             {recall:.4f}
  F1-Score:           {f1:.4f}
  Specificity:        {specificity:.4f}

Stage Pass Rates:
  Stage 1:            {stage1_pass_rate:.2%}
  Stage 2:            {stage2_pass_rate:.2%}
"""
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Final Method Evaluation: Two-Stage Gaussian Filtering with PCA_10D',
            fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

# Save figure
save_path = os.path.join(OUTPUT_DIR, 'final_evaluation_visualization.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Visualization saved to: {save_path}")

## 17. Save Results

In [ ]:
# Save detailed results
results = {
    'method': 'PCA_10D',
    'pca_dim': PCA_DIM,
    'explained_variance': float(explained_var),
    'confidence_stage1': CONFIDENCE_STAGE1,
    'confidence_stage2': CONFIDENCE_STAGE2,
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'specificity': float(specificity)
    },
    'confusion_matrix': {
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    },
    'filtering_statistics': {
        'stage1_pass_count': int(stage1_pass_count),
        'stage2_pass_count': int(stage2_pass_count),
        'stage1_pass_rate': float(stage1_pass_rate),
        'stage2_pass_rate': float(stage2_pass_rate),
        'classified_real': int(np.sum(predictions == 0)),
        'classified_fake': int(np.sum(predictions == 1))
    },
    'dataset': {
        'total_samples': len(test_labels),
        'real_samples': int((test_labels == 0).sum()),
        'fake_samples': int((test_labels == 1).sum())
    }
}

# Save to JSON
results_path = os.path.join(OUTPUT_DIR, 'final_evaluation_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: {results_path}")

# Also save predictions for further analysis
predictions_data = {
    'labels': test_labels.tolist(),
    'predictions': predictions.tolist()
}

predictions_path = os.path.join(OUTPUT_DIR, 'final_predictions.json')
with open(predictions_path, 'w') as f:
    json.dump(predictions_data, f)

print(f"✓ Predictions saved to: {predictions_path}")
print(f"\n{'='*80}")
print(f"EVALUATION COMPLETE")
print(f"{'='*80}")